<a href="https://colab.research.google.com/github/kyungjunoh1/LLM-workspace/blob/main/6_chat_index.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 참고 - 허깅페이스 모델 실행
- Ollama는 추론형 모델이다. 즉, 미세조정할 수 없는 모델이다
- 모델을 파인튜닝하려면 기본 모델을 불러와서 진행해야한다
- https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct
- 중간에 라이센스 동의 후 약 5분 대기. 승인 나면 사용가능
- 승인 후 토큰 생성 -> repo 부분 모두 체크 -> 토큰 로그인 후 사용

In [ ]:
'''
!pip install -U huggingface_hub -qqq
!pip install bitsandbytes==0.47.0 -qqq
!pip install transformers==4.56.1 -qqq

from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import bitsandbytes

login("hf_dQISKCPIkmBKAjGvynefkAkRfxcEkFm")

#양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_id = "meta-llama/Meta-Llama-3-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config
)

model.device

prompt = """
너는 반드시 한국어로만 답변하는 AI다.
영어는 절대 사용하지 마라.

질문: 너는 어떤 모델이야?
답변:
"""
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(
      **inputs,
      max_new_tokens=100,
      do_sample=True,
      temperature=0.7,
      pad_token_id=tokenizer.eos_token_id
    )
answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(answer)
'''

In [ ]:
!ollama --version

In [ ]:
!apt-get update -y
!apt-get install -y curl #curl 설치
!apt-get install -y zstd #zstd : 압축 방식( 올라마 프로그램이 압축되어 있는 형식 )

In [ ]:
#올라마 설치( 올라마 사이트로 받은 파일 코랩에 sh로 설치 )
!curl -fsSL https://ollama.com/install.sh | sh
# 경고는 무시해도 된다.

In [ ]:
!ollama --version

### 코랩 서버에 올라마 서버 구동

In [ ]:
import subprocess
import time

process = subprocess.Popen(["ollama", "serve"])

# 서버가 뜰 시간을 조금 줌
time.sleep(5)

In [ ]:
!ollama --version

In [ ]:
!ollama list

In [ ]:
!ollama pull llama3 #사용할 모델
!ollama pull nomic-embed-text #임베딩 모델

In [ ]:
!ollama list

### 코드 흐름
- 라우터(질문이 policy인지 general인지 고름)
    - route = clean_route(router_chain.invoke({"question": user_question}))
- 검색기(정책 문서를 찾음)
    - nodes = retriever.retrieve(user_question)
- threshold(문서가 충분히 비슷한지 확인)
    - should_use_policy_answer(nodes)
- fallback(문서가 애매하면 그냥 일반 AI 답변)


In [ ]:
!pip install llama-index==0.14.8 -qqq
!pip install llama-index-llms-ollama==0.9.0 -qqq
!pip install llama-index-embeddings-ollama==0.8.4 -qqq
!pip install llama-index-embeddings-ollama==0.8.4 -qqq
!pip install langchain==1.2.15 -qqq
!pip install langchain-community==0.4.1 -qqq

In [ ]:
#from typing import TypedDict, List, Dict, Any
#import os

from typing import List

#LangChain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
#from langchain_huggingface import HuggingFacePipeline

#LlamaIndex
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.core.schema import NodeWithScore
from llama_index.core.postprocessor import SimilarityPostprocessor #유사도 기반 필터링
#from llama_index.embeddings.huggingface import HuggingFaceEmbedding
#from llama_index.llms.huggingface import HuggingFaceLLM


#올라마
from langchain_community.chat_models import ChatOllama
from llama_index.embeddings.ollama import OllamaEmbedding

In [ ]:
llm = ChatOllama(model="llama3", temperature=0.3)

In [ ]:
llm.invoke("한국어로 답해. 너는 어떤 모델이야?")

### 1.라우팅 설정
- prompt 생성
  - policy, general 구분

In [ ]:
router_prompt = ChatPromptTemplate.from_messages([
    ("system", """
너는 사용자 질문을 분류하는 라우터다.

카테고리:
1. policy
- 회사 정책
- 인사/HR
- 연차/휴가/병가
- 근태/출퇴근
- 복지
- 비용처리
- 보안 규정
- 내부 규정
- 사내 프로세스
- 회사 운영 기준

2. general
- 일반 상식
- 개발 지식
- 기술 설명
- 자유 질문


규칙:
1. 반드시 아래 두 단어 중 하나만 출력한다.
   - policy
   - general
2. 설명 추가 금지
3. 이유 설명 금지
4. 문장으로 답변 금지
5. 기호(-, :, 번호) 추가 금지
6. 줄바꿈 추가 금지
7. 애매하지만 회사 제도/내부 기준을 묻는 느낌이면 policy로 출력.
8. 출력 예시:
policy
"""),
    ("user","{question}")
])

In [ ]:
#text = router_prompt.invoke({"question":"연차규정 알려줘"})
#llm.invoke(text)
router_chain = router_prompt | llm | StrOutputParser()

In [ ]:
router_chain.invoke({"question":"오늘의 날씨 알려줘"})

In [ ]:
def clean_route(text:str) -> str:
    text = text.strip().lower()
    if "policy" in text:
        return "policy"
    return "general"

#text = router_chain.invoke({"question":"오늘의 날씨 알려줘"})
#clean_route( text )

### 오케스트레이터
- 이 함수에서 모든 흐름을 제어한다

In [ ]:
user_question = "오늘의 날씨 어때?"

In [ ]:
def run_chat( user_question : str ) -> dict:
  #질문 분류
  route = clean_route( router_chain.invoke({"question": user_question}))
  if route == "policy":
    answer = "문서 답변"
  else:
    answer = "일반 답변"
  return {
      "question":user_question,
      "route":route,
      "answer":answer
  }
run_chat("휴가 신청은 어떻게 진행해?")

### test

In [ ]:
questions = [
    "연차 사용 기준이 뭐야?",
    "병가는 어떻게 써?",
    "딥러닝이 뭐야?",
]
for q in questions:
    result = run_chat(q)
    print("=" * 60)
    print("질문:", result["question"])
    print("route:", result["route"])
    print("답변:", result["answer"])

### 2.일반 답변기능

In [ ]:
general_answer_prompt = ChatPromptTemplate.from_template("""
너는 친절한 AI agent이다.
질문에 대해 한국어로 자연스럽고 이해하기 쉽게 답변해라.

질문:
{question}
""")


In [ ]:
general_chain = general_answer_prompt | llm | StrOutputParser()

In [ ]:
general_chain.invoke({"question":"오늘의 날씨 알려줘"})

In [ ]:
def answer_from_general(question : str) -> str:
  answer = general_chain.invoke({"question":question})
  return answer

### 오케스트레이터 수정

In [ ]:
def run_chat( user_question : str ) -> dict:
  #질문 분류
  route = clean_route( router_chain.invoke({"question": user_question}))
  if route == "policy":
    answer = "문서 답변"
  else:
    answer = answer_from_general( user_question )
  return {
      "question":user_question,
      "route":route,
      "answer":answer
  }
#run_chat("휴가 신청은 어떻게 진행해?")

In [ ]:
questions = [
    "연차 사용 기준이 뭐야?",
    "병가는 어떻게 써?",
    "딥러닝이 뭐야?",
]
for q in questions:
    result = run_chat(q)
    print("=" * 60)
    print("질문:", result["question"])
    print("route:", result["route"])
    print("답변:", result["answer"])

### 3.문서 답변
- 1.문서 파일 로드(txt, json, docs 등...)
- 2.document 로 변환
- 3.백터DB 임베딩 처리로 저장
- 4.유사도 확인 후 사용할 데이터 기반 추론

In [ ]:
!!ollama list

In [ ]:
!pip install llama-index-embeddings-huggingface==0.6.1 -qqq

In [ ]:
#문서에 내용 불러와 document 생성
documents = [
    Document(text="우리 회사의 연차는 1년 만근 시 15일이 부여된다."),
    Document(text="병가는 진단서 제출 시 사용할 수 있으며 최대 30일까지 가능하다."),
    Document(text="출퇴근 기록은 그룹웨어를 통해 등록해야 하며 지각 3회는 경고 대상이다."),
    Document(text="비용 처리는 결제일 기준 7일 이내에 영수증과 함께 등록해야 한다."),
]
#임베딩 설정
embed_model = OllamaEmbedding(model_name="nomic-embed-text") #텍스트 임베딩
#embed_model = HuggingFaceEmbedding( model_name="BAAI/bge-m3" )
#벡터 디비 설정
index = VectorStoreIndex.from_documents(documents, embed_model=embed_model)
#문서에 있는 내용 확인 index 활용
retriever = index.as_retriever(similarity_top_k=3)

In [ ]:
nodes = retriever.retrieve("우리 회사의 연차?")

In [ ]:
for node in nodes:
  print( node )

### 유사도 필터
- 0.85 이상인 데이터만 추출

In [ ]:
post = SimilarityPostprocessor( similarity_cutoff=0.87)
filtered_nodes = post.postprocess_nodes( nodes )
print( len(filtered_nodes ) )
for node in filtered_nodes:
  print(node)

In [ ]:
def retrieve_policy_docs( question : str ) -> tuple:
  nodes = retriever.retrieve( question )

  post = SimilarityPostprocessor( similarity_cutoff=0.87)
  filtered_nodes = post.postprocess_nodes( nodes )

  top_score = 0.0
  if filtered_nodes: #node가 존재한다면
    top_score = filtered_nodes[0].score
  return filtered_nodes, top_score
nodes, score = retrieve_policy_docs("회사 연차관련 내용 알려줘?")

In [ ]:
print(score)
for node in nodes:
  print(node)
print(nodes)

In [ ]:
policy_answer_prompt = ChatPromptTemplate.from_template("""
너는 회사 정책 문서를 바탕으로 답변하는 AI다.

규칙:
1. 답변은 절대적으로 한국어로 자연스럽고 쉽게 작성해라.
2. 아래 [참고 문서]를 기반으로 답변해라.
3. 문서에 없는 내용을 억지로 만들지 마라.
4. 필요한 경우 참고 문서를 짧게 요약해서 설명해라.

[참고 문서]
{context}

[질문]
{question}
""")

In [ ]:
policy_chain = policy_answer_prompt | llm | StrOutputParser()

In [ ]:
def format_nodes( nodes : List[NodeWithScore] ) -> str:
  if not nodes:
    return "검색 문서 없음"
  parts = []
  for i , node in enumerate( nodes, start=1):
    content = node.node.get_content()
    score = node.score if node.score is not None else 0.0

    parts.append(f"[문서{i}] | score={score:.4f}]\n{content}")
  return "\n\n".join(parts)
txt = format_nodes(nodes)

In [ ]:
policy_chain.invoke({"context":txt, "question":"회사 연차관련 내용 알려줘?"})

In [ ]:
def answer_from_policy( question : str, nodes : List[NodeWithScore]) -> str:
  context = format_nodes( nodes ) #문서를 하나로 합치는 기능
  answer = policy_chain.invoke({"context":context, "question":question}) #추론
  return f"[자료 기반 답변]\n{answer}"

In [ ]:
question = "회사 연차관련 내용 알려줘?"
answer_from_policy(question, nodes)

### 최종 수정 오케스트레이터

In [ ]:
def run_chat( user_question : str ) -> dict:
  #질문 분류
  route = clean_route( router_chain.invoke({"question": user_question}))

  final_route = route
  top_score = 0.0

  if route == "policy":
    nodes, top_score = retrieve_policy_docs( user_question )
    if nodes: #값이 있으면 문서 llm 실행
      answer = answer_from_policy( user_question , nodes )
    #fallback(대체수단)
    #llm 추론은 policy로 했지만 실제 문서 유사도가 낮아 문서에 없음으로 판단 일반 llm연결
    else:
      answer = answer_from_general( user_question )
      final_route = "general"
    #answer = "문서 답변"
  else:
    answer = answer_from_general( user_question )
  return {
      "question":user_question,
      "route":route,
      "final_route" : final_route,
      "top_score" : top_score,
      "answer":answer
  }
#run_chat("휴가 신청은 어떻게 진행해?")

In [ ]:
questions = [
    "연차 사용 기준이 뭐야?",
    "병가는 어떻게 써?",
    "딥러닝이 뭐야?",
]
for q in questions:
    result = run_chat(q)
    print("=" * 60)
    print("질문:", result["question"])
    print("route:", result["route"])
    print("답변:", result["answer"])